In [ ]:
!uv pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

In [1]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
from tribev2 import TribeModel
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Model Loaded.")

/usr/local/lib/python3.12/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-04-04 00:32:42 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


config.yaml: 0.00B [00:00, ?B/s]

best.ckpt:   0%|          | 0.00/709M [00:00<?, ?B/s]

2026-04-04 00:32:46 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
INFO:tribev2.demo_utils:Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.12/dist-packages/x_transformers/x_transformers.py:461: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use 

Model Loaded.


In [3]:
# 1. TRIMODAL (full experience)
df = model.get_events_dataframe(video_path="clip.mp4")

Extract audio from video events:   0%|          | 0/1 [00:00<?, ?it/s]

MoviePy - Writing audio in clip.wav



Extract audio from video events: 100%|██████████| 1/1 [00:02<00:00,  2.23s/it]
/usr/local/lib/python3.12/dist-packages/neuralset/events/transforms/audio.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  events = pd.concat([events, pd.DataFrame(events_to_add)], ignore_index=True)
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)


MoviePy - Done.


Extracting words from audio: 100%|██████████| 1/1 [01:35<00:00, 95.54s/it]
/usr/local/lib/python3.12/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Add context to words: 100%|██████████| 2160/2160 [00:00<00:00, 54522.62it/s]


In [4]:
print(df.columns.tolist())
print(df["type"].unique())
print(df.head(20))

['type', 'start', 'duration', 'timeline', 'subject', 'session', 'task', 'run', 'filepath', 'frequency', 'offset', 'stop', 'text', 'sequence_id', 'sentence', 'language', 'sentence_char', 'text_char', 'context', 'modality']
['Audio' 'Video' 'Sentence' 'Text' 'Word']
        type      start    duration timeline  subject session task run  \
0      Audio   0.000000   60.000000  default  default                    
1      Video   0.000000   60.000000  default  default                    
2   Sentence   8.285999    0.600002  default  default                    
3       Text   8.286000  785.895000  default  default                    
4       Word   8.286000    0.040000  default  default                    
5       Word   8.346000    0.180000  default  default                    
6       Word   8.566000    0.320000  default  default                    
7   Sentence  10.306999    0.841002  default  default                    
8       Word  10.307000    0.141000  default  default                

In [10]:
import json
import pandas as pd
import numpy as np

with open("transcript.json") as f:
    transcript = json.load(f)

# Keep Audio and Video rows from TRIBE's original extraction
df_av = df[df["type"].isin(["Audio", "Video"])].copy()

# Build Word rows with context
word_rows = []
all_words_so_far = []

for seg in transcript["segments"]:
    for w in seg.get("words", []):
        word_text = w["word"].strip()
        dur = w["end"] - w["start"]
        if dur <= 0:
            dur = 0.01

        all_words_so_far.append(word_text)
        context = " ".join(all_words_so_far[-1024:])

        word_rows.append({
            "type": "Word",
            "start": w["start"],
            "duration": dur,
            "stop": w["end"],
            "text": word_text,
            "sentence": seg["text"].strip(),
            "sequence_id": float(transcript["segments"].index(seg)),
            "language": "english",
            "context": context,
            "timeline": "default",
            "subject": "default",
            "session": "",
            "task": "",
            "run": "",
        })

# Build Sentence rows
sentence_rows = []
for i, seg in enumerate(transcript["segments"]):
    dur = seg["end"] - seg["start"]
    if dur <= 0:
        dur = 0.01
    sentence_rows.append({
        "type": "Sentence",
        "start": seg["start"],
        "duration": dur,
        "stop": seg["end"],
        "text": seg["text"].strip(),
        "timeline": "default",
        "subject": "default",
        "session": "",
        "task": "",
        "run": "",
    })

# Build the single Text row
text_row = {
    "type": "Text",
    "start": transcript["segments"][0]["start"],
    "duration": transcript["segments"][-1]["end"] - transcript["segments"][0]["start"],
    "stop": transcript["segments"][-1]["end"],
    "text": transcript["text"].strip(),
    "sequence_id": 0.0,
    "language": "english",
    "timeline": "default",
    "subject": "default",
    "session": "",
    "task": "",
    "run": "",
}

# Combine
df_corrected = pd.concat([
    df_av,
    pd.DataFrame([text_row]),
    pd.DataFrame(sentence_rows),
    pd.DataFrame(word_rows),
], ignore_index=True)

df_corrected = df_corrected.sort_values("start").reset_index(drop=True)
print(f"Corrected dataframe: {len(df_corrected)} rows")
print(df_corrected["type"].value_counts())

Corrected dataframe: 542 rows
type
Word        436
Sentence     95
Audio         5
Video         5
Text          1
Name: count, dtype: int64


In [11]:
preds_tri, _ = model.predict(events=df_corrected)
np.save("preds_trimodal.npy", preds_tri)
print(f"Trimodal: {preds_tri.shape}")

[01:06:28 INFO] Preparing extractor: text
INFO:tribev2.main:Preparing extractor: text

  0%|          | 0/436 [00:00<?, ?it/s]

Computing word embeddings:   0%|          | 0/109 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

2026-04-04 01:07:55 - DEBUG - neuralset.extractors.video:277 - Loaded Video (duration 60.0s at 24.0fps, shape (1920, 1080)):
clip.mp4
DEBUG:neuralset.extractors.video:Loaded Video (duration 60.0s at 24.0fps, shape (1920, 1080)):
clip.mp4


Encoding video:   0%|          | 0/120 [00:00<?, ?it/s]2026-04-04 01:07:59 - DEBUG - neuralset.extractors.video:311 - Created Tensor with size (120, 20, 1408)
DEBUG:neuralset.extractors.video:Created Tensor with size (120, 20, 1408)


Encoding video:   1%|          | 1/120 [00:04<08:00,  4.04s/it]

Encoding video:   2%|▏         | 2/120 [00:08<08:12,  4.18s/it]

Encoding video:   2%|▎         | 3/120 [00:12<07:50,  4.02s/it]

Encoding video:   3%|▎         | 4/120 [00:16<07:44,  4.00s/it]

Encoding video:   4%|▍         | 5/120 [00:20<07:47,  4.07s/it]

Encoding video:   5%|▌         | 6/120 [00:24<07:42,  4.06s/it]

Encoding video:   6%|▌         | 7/120 [00:28<07:38,  4.06s/it]

Encoding video:   7%|▋         | 8/120 [00:32<07:42,  4.13s/it]

Encod

Trimodal: (325, 20484)


In [12]:
# Cell 3 — VISUAL ONLY
df_visual = df_corrected[df_corrected["type"] == "Video"].reset_index(drop=True)
preds_vis, _ = model.predict(events=df_visual)
np.save("preds_visual.npy", preds_vis)
print(f"Visual: {preds_vis.shape}")

[02:15:08 WARNING] Removing extractor text as there are no corresponding events
[02:15:08 WARNING] Removing extractor audio as there are no corresponding events
[02:15:08 INFO] Preparing extractor: video
INFO:tribev2.main:Preparing extractor: video
[02:15:08 INFO] Preparing extractor: subject_id
INFO:tribev2.main:Preparing extractor: subject_id
2026-04-04 02:15:08 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[02:15:08 INFO] Building dataloader for split all
INFO:tribev2.main:Building dataloader for split all
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid pot

Visual: (325, 20484)


In [13]:
# Cell 4 — ACOUSTIC ONLY
df_acoustic = df_corrected[df_corrected["type"] == "Audio"].reset_index(drop=True)
preds_aco, _ = model.predict(events=df_acoustic)
np.save("preds_acoustic.npy", preds_aco)
print(f"Acoustic: {preds_aco.shape}")

[02:15:55 WARNING] Removing extractor text as there are no corresponding events
[02:15:55 WARNING] Removing extractor video as there are no corresponding events
[02:15:55 INFO] Preparing extractor: audio
INFO:tribev2.main:Preparing extractor: audio
[02:15:56 INFO] Preparing extractor: subject_id
INFO:tribev2.main:Preparing extractor: subject_id
2026-04-04 02:15:56 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[02:15:56 INFO] Building dataloader for split all
INFO:tribev2.main:Building dataloader for split all
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid pot

Acoustic: (325, 20484)


In [14]:
# Cell 5 — SEMANTIC ONLY
df_semantic = df_corrected[df_corrected["type"].isin(["Sentence", "Text", "Word"])].reset_index(drop=True)
preds_sem, _ = model.predict(events=df_semantic)
np.save("preds_semantic.npy", preds_sem)
print(f"Semantic: {preds_sem.shape}")

[02:16:42 WARNING] Removing extractor video as there are no corresponding events
[02:16:42 WARNING] Removing extractor audio as there are no corresponding events
[02:16:42 INFO] Preparing extractor: text
INFO:tribev2.main:Preparing extractor: text
[02:16:42 INFO] Preparing extractor: subject_id
INFO:tribev2.main:Preparing extractor: subject_id
2026-04-04 02:16:42 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[02:16:42 INFO] Building dataloader for split all
INFO:tribev2.main:Building dataloader for split all
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid pote

Semantic: (307, 20484)


In [15]:
# Cell 6 — AUDIO + VISUAL
df_av_only = df_corrected[df_corrected["type"].isin(["Audio", "Video"])].reset_index(drop=True)
preds_av, _ = model.predict(events=df_av_only)
np.save("preds_audvis.npy", preds_av)
print(f"Audio+Visual: {preds_av.shape}")

[02:17:49 WARNING] Removing extractor text as there are no corresponding events
[02:17:49 INFO] Preparing extractor: audio
INFO:tribev2.main:Preparing extractor: audio
[02:17:49 INFO] Preparing extractor: video
INFO:tribev2.main:Preparing extractor: video
[02:17:49 INFO] Preparing extractor: subject_id
INFO:tribev2.main:Preparing extractor: subject_id
2026-04-04 02:17:49 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[02:17:50 INFO] Building dataloader for split all
INFO:tribev2.main:Building dataloader for split all
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to av

Audio+Visual: (325, 20484)


In [16]:
# Cell 7 — AUDIO + TEXT
df_at = df_corrected[df_corrected["type"] != "Video"].reset_index(drop=True)
preds_at, _ = model.predict(events=df_at)
np.save("preds_audtext.npy", preds_at)
print(f"Audio+Text: {preds_at.shape}")

[02:19:13 WARNING] Removing extractor video as there are no corresponding events
[02:19:13 INFO] Preparing extractor: text
INFO:tribev2.main:Preparing extractor: text
[02:19:13 INFO] Preparing extractor: audio
INFO:tribev2.main:Preparing extractor: audio
[02:19:13 INFO] Preparing extractor: subject_id
INFO:tribev2.main:Preparing extractor: subject_id
2026-04-04 02:19:13 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[02:19:13 INFO] Building dataloader for split all
INFO:tribev2.main:Building dataloader for split all
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avo

Audio+Text: (325, 20484)


In [17]:
# Cell 8 — VISUAL + TEXT
df_vt = df_corrected[df_corrected["type"] != "Audio"].reset_index(drop=True)
preds_vt, _ = model.predict(events=df_vt)
np.save("preds_vistext.npy", preds_vt)
print(f"Visual+Text: {preds_vt.shape}")

[02:19:54 WARNING] Removing extractor audio as there are no corresponding events
[02:19:54 INFO] Preparing extractor: text
INFO:tribev2.main:Preparing extractor: text
[02:19:54 INFO] Preparing extractor: video
INFO:tribev2.main:Preparing extractor: video
[02:19:54 INFO] Preparing extractor: subject_id
INFO:tribev2.main:Preparing extractor: subject_id
2026-04-04 02:19:54 - WARNING - neuralset.extractors.base:824 - LabelEncoder has only found one label: {'default'}. This was probably not intended.
[02:19:54 INFO] Building dataloader for split all
INFO:tribev2.main:Building dataloader for split all
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avo

Visual+Text: (325, 20484)
